# Seasonal Stage 1 – Stagel 3‑Month Diagnostics

This notebook summarizes and visualizes outputs for a single run:
- seasonal vs base storage activity
- load shedding and stress periods
- curtailment and generation signals


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")


In [ ]:
# --- User configuration -----------------------------------------------------
PROJECT_DIR = Path("/home/fs01/jl2966/acorn-julia2/acorn-julia")
RUN_NAME = "low_RE_mod_elec_iter0"
CLIMATE_SCENARIO = "historical_1980_2019"
SAVE_NAME = "stagel_3month"
YEAR = 1985

run_dir = PROJECT_DIR / "runs" / RUN_NAME / "outputs" / CLIMATE_SCENARIO / SAVE_NAME
print("Run dir:", run_dir)


In [ ]:
# --- Helpers ----------------------------------------------------------------

def tidy_storage_df(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    meta_cols = [c for c in ["bus_id", "asset_type", "zone", "is_seasonal"] if c in df.columns]
    value_cols = [c for c in df.columns if c not in meta_cols]
    tidy = df.melt(id_vars=meta_cols, value_vars=value_cols,
                   var_name="timestamp", value_name=value_name)
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"], errors="coerce")
    tidy = tidy.dropna(subset=["timestamp"])
    return tidy


def tidy_bus_csv(path: Path, value_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    meta_cols = [c for c in ["bus_id", "zone"] if c in df.columns]
    value_cols = [c for c in df.columns if c not in meta_cols]
    tidy = df.melt(id_vars=meta_cols, value_vars=value_cols,
                   var_name="timestamp", value_name=value_name)
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"], errors="coerce")
    tidy = tidy.dropna(subset=["timestamp"])
    return tidy


def total_timeseries(df: pd.DataFrame, value_name: str) -> pd.Series:
    return df.groupby("timestamp")[value_name].sum()


def event_stats(series: pd.Series, threshold: float = 1e-6) -> pd.DataFrame:
    s = series.fillna(0.0)
    events = []
    in_event = False
    start = None
    total = 0.0
    hours = 0
    for ts, val in s.items():
        if val > threshold:
            if not in_event:
                in_event = True
                start = ts
                total = 0.0
                hours = 0
            total += float(val)
            hours += 1
            end = ts
        else:
            if in_event:
                events.append((start, end, hours, total))
                in_event = False
    if in_event:
        events.append((start, end, hours, total))

    if not events:
        return pd.DataFrame(columns=["start", "end", "hours", "total_MWh"])
    df = pd.DataFrame(events, columns=["start", "end", "hours", "total_MWh"])
    return df.sort_values("hours", ascending=False)


In [ ]:
# --- Load outputs -----------------------------------------------------------
charge = tidy_storage_df(pd.read_csv(run_dir / f"charge_{YEAR}.csv"), "charge")
discharge = tidy_storage_df(pd.read_csv(run_dir / f"discharge_{YEAR}.csv"), "discharge")
charge_base = tidy_storage_df(pd.read_csv(run_dir / f"charge_base_{YEAR}.csv"), "charge")
discharge_base = tidy_storage_df(pd.read_csv(run_dir / f"discharge_base_{YEAR}.csv"), "discharge")
charge_seasonal = tidy_storage_df(pd.read_csv(run_dir / f"charge_seasonal_{YEAR}.csv"), "charge")
discharge_seasonal = tidy_storage_df(pd.read_csv(run_dir / f"discharge_seasonal_{YEAR}.csv"), "discharge")

storage_state = tidy_storage_df(pd.read_csv(run_dir / f"storage_state_{YEAR}.csv"), "storage_state")
load_shed = tidy_bus_csv(run_dir / f"load_shedding_{YEAR}.csv", "load_shedding")
wind_curt = tidy_bus_csv(run_dir / f"wind_curtailment_{YEAR}.csv", "wind_curtailment")
solar_curt = tidy_bus_csv(run_dir / f"solar_curtailment_{YEAR}.csv", "solar_curtailment")

print("Loaded outputs for year", YEAR)


In [ ]:
# --- Quick summary -----------------------------------------------------------
summary = {
    "load_shed_MWh": load_shed["load_shedding"].sum(),
    "charge_MWh": charge["charge"].sum(),
    "discharge_MWh": discharge["discharge"].sum(),
    "base_discharge_MWh": discharge_base["discharge"].sum(),
    "seasonal_discharge_MWh": discharge_seasonal["discharge"].sum(),
}

display(pd.DataFrame([summary]))


In [ ]:
# --- Fleet activity (base vs seasonal) -------------------------------------
base_charge_ts = total_timeseries(charge_base, "charge")
base_discharge_ts = total_timeseries(discharge_base, "discharge")
seasonal_charge_ts = total_timeseries(charge_seasonal, "charge")
seasonal_discharge_ts = total_timeseries(discharge_seasonal, "discharge")
load_shed_ts = total_timeseries(load_shed, "load_shedding")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(base_charge_ts.index, base_charge_ts.values, label="base charge", alpha=0.6)
axes[0].plot(base_discharge_ts.index, -base_discharge_ts.values, label="base discharge", alpha=0.6)
axes[0].plot(seasonal_charge_ts.index, seasonal_charge_ts.values, label="seasonal charge", alpha=0.6)
axes[0].plot(seasonal_discharge_ts.index, -seasonal_discharge_ts.values, label="seasonal discharge", alpha=0.6)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_ylabel("MW")
axes[0].set_title("Storage fleet activity")
axes[0].legend(loc="upper right")

axes[1].plot(load_shed_ts.index, load_shed_ts.values, color="C3", label="load shedding")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("MW")
axes[1].set_title("Load shedding")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
# --- Storage usage by asset type --------------------------------------------
if "asset_type" in discharge.columns:
    by_type = discharge.groupby("asset_type")["discharge"].sum().sort_values(ascending=False)
    display(by_type.to_frame("discharge_MWh"))

    by_type_charge = charge.groupby("asset_type")["charge"].sum().sort_values(ascending=False)
    display(by_type_charge.to_frame("charge_MWh"))


In [ ]:
# --- Curtailment overview ----------------------------------------------------
wind_curt_ts = total_timeseries(wind_curt, "wind_curtailment")
solar_curt_ts = total_timeseries(solar_curt, "solar_curtailment")

fig, ax = plt.subplots(1, 1, figsize=(14, 4))
ax.plot(wind_curt_ts.index, wind_curt_ts.values, label="wind curtailment", alpha=0.7)
ax.plot(solar_curt_ts.index, solar_curt_ts.values, label="solar curtailment", alpha=0.7)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("MW")
ax.set_title("Renewable curtailment")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
# --- Load shedding stress events -------------------------------------------
ls_events = event_stats(load_shed_ts, threshold=0.0)
print("Top load shedding events (hours, MWh):")
display(ls_events.head(10))

print("Total load shedding hours:", int((load_shed_ts > 0).sum()))
print("Peak load shedding MW:", float(load_shed_ts.max()))


In [ ]:
# --- Seasonal usage diagnostics ---------------------------------------------
seasonal_hours = int((seasonal_discharge_ts > 0).sum())
print("Seasonal discharge hours:", seasonal_hours)
print("Seasonal discharge MWh:", seasonal_discharge_ts.sum())

if seasonal_hours == 0:
    print("Seasonal storage not used. Potential reasons: rate caps too tight, no shortage at seasonal buses, or base storage covers all shortages.")


In [ ]:
# --- Check seasonal capacity + rate caps ------------------------------------
debug_path = run_dir / "_storage_debug.csv"
if debug_path.exists():
    debug = pd.read_csv(debug_path)
    if "is_seasonal" in debug.columns:
        seasonal = debug[debug["is_seasonal"] == 1].copy()
        if not seasonal.empty:
            total_energy = seasonal["storage_capacity_mwh"].sum()
            total_power = seasonal["charge_capacity_MW"].sum()
            if "max_discharge_rate_frac" in seasonal.columns:
                max_discharge = (seasonal["max_discharge_rate_frac"] * seasonal["storage_capacity_mwh"]).sum()
            else:
                max_discharge = np.nan
            print("Seasonal rows:", len(seasonal))
            print("Total seasonal energy (MWh):", total_energy)
            print("Total seasonal charge cap (MW):", total_power)
            print("Estimated max seasonal discharge (MW):", max_discharge)
            print("Peak load shedding (MW):", float(load_shed_ts.max()))
